# Graph adjacency and mesh correspondence checks

**Role:** Analysis.

Inspect the correspondence among PyG graphs, NetworkX neighborhoods, cell coordinates and mesh projections. Use selected source organoids to check edge construction, node IDs and spatial mapping before interpreting model outputs. The notebook contains interactive visual checks; automated numerical assertions live in tests/. It does not train models or alter the dataset.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Data and analysis settings

Select the dataset or saved run and the analysis options below. Model-dependent analyses restore the recorded inputs and fitted transformations before computing results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import networkx as nx
import sys
from pathlib import Path

PROJECT_ROOT = PROJECT_ROOT  # Locate the repository from the notebook’s working directory.
DATA_ROOT = PROJECT_ROOT / "training_data"  # Parent directory containing source datasets.

sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)

## Load graphs and select exemplars

Read the graph datasets and metadata needed for an explicit set of organoid examples.

In [ ]:
from src.data.io import load_graph_dataset_from_dir
from src.data.metadata import load_aux_metadata_for_dir, attach_metadata_to_graphs, load_marker_names_from_dir
from src.data.splits import graph_metadata_key, select_graphs_by_keys

DEFAULT_MARKER_COLORS = {
    "Lysozyme": "#2C75D2",  # Display color for Lysozyme.
    "Serotonin": "#F392E3",  # Display color for Serotonin.
    "Mucin 2": "#3EC1D9",  # Display color for Mucin 2.
    "Chroma": "#D852CB",  # Display color for Chroma.
    "Glucagon": "#983EE2",  # Display color for Glucagon.
    "LGR5": "#FFB431",  # Display color for LGR5.
    "AldoB": "#F16C6A",  # Display color for AldoB.
    "Agr2": "#359BD5",  # Display color for Agr2.
    "KI67": "#808080",  # Display color for KI67.
    "none": "#EBEBEB",  # Display color for none.
}  # Display colors for cell identities; not model inputs.

forced_keys = {
    ("20251201", "day4p5_B03_144"),
    ("20251201", "day4p5_B02_124"),
    ("20251201", "day4p5_B04_4"),
    ("20251201", "day4p5_B02_115"),
}  # Stable dataset/organoid keys chosen for manual inspection.

In [ ]:
data_dir = PROJECT_ROOT / "training_data" / "smooth_cryptdist"  # Dataset directory supplying graphs and metadata.

graphs = load_graph_dataset_from_dir(str(data_dir))
print(f"Loaded {len(graphs)} organoids.")

meta = load_aux_metadata_for_dir(str(data_dir))
attach_metadata_to_graphs(graphs, meta)

marker_names = load_marker_names_from_dir(data_dir)
print(marker_names)

## Select corresponding PyG organoids

Resolve stable organoid keys to the PyG graphs used in the adjacency and mesh comparisons.

In [ ]:
selected_pyg_graphs = select_graphs_by_keys(
    graphs,
    forced_keys,
    key_fn=graph_metadata_key,
    inplace=False,
)

print(f"Selected {len(selected_pyg_graphs)} PyG graphs matching forced_keys.")

# Optional safety check
selected_keys = [graph_metadata_key(g) for g in selected_pyg_graphs]
print("Selected keys:")
for k in selected_keys:
    print("  ", k)


## Render original graph fate labels

Use the source graph representation and marker colors to establish the visual reference.

In [ ]:
import organograph
from organograph.plotting.graphs import plot_graph_by_markers

marker_map = [
    {"marker": name, "color": color, "name": name}
    for name, color in DEFAULT_MARKER_COLORS.items()
    if name != "none" and name in marker_names
]

def get_node_arrays(G):
    nodes_sorted = sorted(G.nodes())
    centroids = np.array([np.asarray(G.nodes[n]["centroid"], dtype=float) for n in nodes_sorted])

    feat_key = None
    for k in ("marker_bin", "markers_bin", "x"):
        if k in G.nodes[nodes_sorted[0]]:
            feat_key = k
            break
    if feat_key is None:
        raise KeyError("Could not find node feature key among: 'marker_bin', 'markers_bin', 'x'")

    X = np.array([np.asarray(G.nodes[n][feat_key], dtype=float) for n in nodes_sorted])
    return nodes_sorted, centroids, X, feat_key

def alignment_qc(G_nx, G_pyg_nx):
    nx_nodes, nx_centroids, nx_X, nx_feat_key = get_node_arrays(G_nx)
    pyg_nodes, pyg_centroids, pyg_X, pyg_feat_key = get_node_arrays(G_pyg_nx)

    out = {
        "nx_num_nodes": G_nx.number_of_nodes(),
        "pyg_num_nodes": G_pyg_nx.number_of_nodes(),
        "nx_num_edges": G_nx.number_of_edges(),
        "pyg_num_edges": G_pyg_nx.number_of_edges(),
        "nx_feat_key": nx_feat_key,
        "pyg_feat_key": pyg_feat_key,
    }

    if nx_X.shape[1] != pyg_X.shape[1]:
        out["feature_dim_match"] = False
        out["nx_feat_dim"] = nx_X.shape[1]
        out["pyg_feat_dim"] = pyg_X.shape[1]
        return out

    d2 = ((pyg_centroids[:, None, :] - nx_centroids[None, :, :]) ** 2).sum(axis=2)
    nn_idx = np.argmin(d2, axis=1)
    nn_dist = np.sqrt(d2[np.arange(len(pyg_nodes)), nn_idx])
    feat_match = (pyg_X.astype(int) == nx_X[nn_idx].astype(int)).all(axis=1)

    out["feature_dim_match"] = True
    out["median_centroid_distance"] = float(np.median(nn_dist))
    out["mean_centroid_distance"] = float(np.mean(nn_dist))
    out["max_centroid_distance"] = float(np.max(nn_dist))
    out["exact_marker_match_fraction"] = float(np.mean(feat_match))
    out["exact_marker_match_count"] = int(np.sum(feat_match))
    out["num_pyg_nodes"] = int(len(pyg_nodes))

    return out

def show_graph_pair_plotly(G_nx, G_pyg_nx, title_prefix="", fig_size=(750, 550)):
    fig1 = plot_graph_by_markers(
        G_nx,
        marker_map,
        backend="plotly",
        baseline_color=DEFAULT_MARKER_COLORS["none"],
        node_size=5,
        edge_width=1.0,
        priority="first",
        add_legend=True,
        legend_baseline_name="none",
        hide_axes=True,
        transparent_bg=True,
        fig_size=fig_size,
    )
    fig1.update_layout(title=f"{title_prefix}Original NetworkX graph")
    fig1.show()

    fig2 = plot_graph_by_markers(
        G_pyg_nx,
        marker_map,
        backend="plotly",
        baseline_color=DEFAULT_MARKER_COLORS["none"],
        node_size=5,
        edge_width=1.0,
        priority="first",
        add_legend=True,
        legend_baseline_name="none",
        hide_axes=True,
        transparent_bg=True,
        fig_size=fig_size,
    )
    fig2.update_layout(title=f"{title_prefix}PyG graph converted to NetworkX")
    fig2.show()


## Compare NetworkX and PyG graph structure

Match source and converted graph nodes and inspect neighborhood structure and cell coordinates.

In [ ]:
from organograph.graph.io import load_cell_graph
from src.plotting.motif_plots import pyg_graph_to_nx_with_centroids

paired_graphs = []

for pyg_graph in selected_pyg_graphs:
    md = getattr(pyg_graph, "meta", None)
    if not isinstance(md, dict):
        raise ValueError(f"{getattr(pyg_graph, 'organoid_str', None)} has no attached metadata dict")

    graph_path = md.get("graph_path", None)
    if graph_path is None:
        raise ValueError(f"{getattr(pyg_graph, 'organoid_str', None)} missing meta['graph_path']")

    # Load original segmentation graph
    G_nx = load_cell_graph(graph_path)

    # Convert PyG graph to NetworkX with centroids reconstructed from mesh metadata
    G_pyg_nx = pyg_graph_to_nx_with_centroids(
        pyg_graph,
        marker_names=marker_names,
        meta_lookup=None,
        normalize_mesh=True,
    )

    paired_graphs.append(
        {
            "key": graph_metadata_key(pyg_graph),
            "organoid_str": getattr(pyg_graph, "organoid_str", None),
            "graph_path": graph_path,
            "pyg_graph": pyg_graph,
            "nx_graph": G_nx,
            "pyg_as_nx_graph": G_pyg_nx,
        }
    )

# ---------------------------------------------------------------------
# 7) QC + plotting
# ---------------------------------------------------------------------
for item in paired_graphs:
    key = item["key"]
    organoid_str = item["organoid_str"]
    G_nx = item["nx_graph"]
    G_pyg_nx = item["pyg_as_nx_graph"]

    qc = alignment_qc(G_nx, G_pyg_nx)

    print("=" * 80)
    print(f"Key:          {key}")
    print(f"Organoid:     {organoid_str}")
    print(f"NX nodes:     {qc['nx_num_nodes']}")
    print(f"PyG nodes:    {qc['pyg_num_nodes']}")
    print(f"NX edges:     {qc['nx_num_edges']}")
    print(f"PyG edges:    {qc['pyg_num_edges']}")
    print(f"NX feat key:  {qc['nx_feat_key']}")
    print(f"PyG feat key: {qc['pyg_feat_key']}")

    if qc["feature_dim_match"]:
        print(f"Median centroid distance: {qc['median_centroid_distance']:.6f}")
        print(f"Mean centroid distance:   {qc['mean_centroid_distance']:.6f}")
        print(f"Max centroid distance:    {qc['max_centroid_distance']:.6f}")
        print(
            f"Exact marker-vector match: "
            f"{qc['exact_marker_match_count']} / {qc['num_pyg_nodes']} "
            f"({100 * qc['exact_marker_match_fraction']:.2f}%)"
        )
    else:
        print(
            f"Feature dimension mismatch: "
            f"NX={qc['nx_feat_dim']} vs PyG={qc['pyg_feat_dim']}"
        )

    show_graph_pair_plotly(
        G_nx,
        G_pyg_nx,
        title_prefix=f"{organoid_str} | ",
        fig_size=(800, 600),
    )

## Mesh projection helpers

Define the local diagnostic views used to compare categorical node labels with their projected mesh vertices.

In [ ]:
import numpy as np
from organograph.plotting.meshes import plot_mesh_by_regions
from src.plotting.mesh_plots import project_node_categories_to_mesh

def plot_marker_mesh_consistent(
    graph,
    *,
    marker_names,           # must match graph.x column order exactly
    marker_colors,          # dict; order defines priority, same as graph plot
    meta_lookup=None,
    threshold=0.5,
    fig_size=(800, 700),
    view=None,
    normalize_mesh=True,
    add_legend=True,
):
    """
    Mesh marker plot consistent with plot_graph_by_markers(..., priority="first").

    Behavior:
    - marker priority follows marker_colors dict order, exactly like the graph plot
    - cells with no positive marker become "none"
    - mesh vertices with vertex_owner < 0 are also forced to "none"
    """

    X = graph.x.detach().cpu().numpy()

    if X.shape[1] != len(marker_names):
        raise ValueError(
            f"graph.x has shape {X.shape}, but marker_names has length {len(marker_names)}. "
            "marker_names must match graph.x column order exactly."
        )

    # Build name -> column index from the TRUE feature-column order
    marker_to_col = {name: j for j, name in enumerate(marker_names)}

    # Use the SAME priority order as the graph plot:
    # the order of marker_colors / DEFAULT_MARKER_COLORS
    priority_markers = [m for m in marker_colors.keys() if m != "none" and m in marker_to_col]

    # Assign one category per node
    node_categories = np.full(X.shape[0], "none", dtype=object)
    assigned = np.zeros(X.shape[0], dtype=bool)

    for m in priority_markers:
        j = marker_to_col[m]
        pos = X[:, j] > threshold
        take = pos & (~assigned)
        node_categories[take] = m
        assigned[take] = True

    # Project to mesh
    proj = project_node_categories_to_mesh(
        graph,
        node_categories,
        meta_lookup=meta_lookup,
        normalize_mesh=normalize_mesh,
    )

    mesh = proj["mesh"]
    mesh_categories = np.asarray(proj["mesh_categories"], dtype=object)

    # IMPORTANT: collapse uncovered mesh vertices into "none"
    mesh_categories[mesh_categories == None] = "none"

    # Region order follows same graph priority + none last
    category_order = priority_markers + ["none"]

    regions = []
    region_names = []
    colors = []

    for cat in category_order:
        idx = np.where(mesh_categories == cat)[0]
        if idx.size == 0:
            continue
        regions.append(idx)
        region_names.append(cat)
        colors.append(marker_colors[cat])

    fig = plot_mesh_by_regions(
        mesh,
        regions,
        backend="plotly",
        region_kind="vertex",
        region_names=region_names,
        colors=colors,
        baseline_color=marker_colors["none"],
        baseline_name="none",
        priority="last",
        alpha=1.0,
        view=view,
        fig_size=fig_size,
        add_legend=add_legend,
    )

    fig.update_layout(title=getattr(graph, "organoid_str", "marker projection"))
    return fig

def debug_marker_projection(graph, marker_names, marker_colors, threshold=0.5):
    X = graph.x.detach().cpu().numpy()
    print("graph.x shape:", X.shape)
    print("marker_names length:", len(marker_names))
    print("marker_names:", marker_names)
    print("priority from marker_colors:", [k for k in marker_colors.keys() if k != "none"])

    # node categories using marker_names order (old mesh behavior)
    old_node_categories = []
    for row in X:
        label = "none"
        for j, name in enumerate(marker_names):
            if row[j] > threshold:
                label = name
                break
        old_node_categories.append(label)
    old_node_categories = np.asarray(old_node_categories, dtype=object)

    # node categories using marker_colors order (graph-consistent behavior)
    marker_to_col = {name: j for j, name in enumerate(marker_names)}
    priority_markers = [m for m in marker_colors.keys() if m != "none" and m in marker_to_col]
    new_node_categories = np.full(X.shape[0], "none", dtype=object)
    assigned = np.zeros(X.shape[0], dtype=bool)
    for m in priority_markers:
        j = marker_to_col[m]
        pos = X[:, j] > threshold
        take = pos & (~assigned)
        new_node_categories[take] = m
        assigned[take] = True

    print("\nCounts with marker_names-order assignment:")
    for c in np.unique(old_node_categories):
        print(f"  {c}: {(old_node_categories == c).sum()}")

    print("\nCounts with graph-consistent assignment:")
    for c in np.unique(new_node_categories):
        print(f"  {c}: {(new_node_categories == c).sum()}")

    print("\nNumber of cells whose assigned label changes:",
          np.sum(old_node_categories != new_node_categories))

    proj = project_node_categories_to_mesh(graph, new_node_categories, meta_lookup=None)
    vertex_owner = np.asarray(proj["vertex_owner"])
    print("Uncovered mesh vertices (vertex_owner < 0):", np.sum(vertex_owner < 0))
    print("Covered mesh vertices:", np.sum(vertex_owner >= 0))

## Inspect node-to-mesh correspondence

Display the mapping for each selected organoid, checking marker priority and ownership arrays on the same cells.

In [ ]:
for g in selected_pyg_graphs:
    debug_marker_projection(
        g,
        marker_names=marker_names,
        marker_colors=DEFAULT_MARKER_COLORS,
        threshold=0.5,
    )

## Render the consistent marker mesh

Use the validated correspondence to produce the final marker-colored mesh views.

In [ ]:
for g in selected_pyg_graphs:
    fig = plot_marker_mesh_consistent(
        g,
        marker_names=marker_names,
        marker_colors=DEFAULT_MARKER_COLORS,
        meta_lookup=None,
        threshold=0.5,
        fig_size=(900, 750),
    )
    fig.show()